# Time Series Preprocessing and Fitting Tutorial

This notebook provides a comprehensive guide to preprocessing real-world time series data before fitting innovation diffusion models. We'll cover data cleaning, trend extraction, seasonality handling, noise reduction, and robust fitting strategies.

## Table of Contents

1. **Introduction to Time Series Preprocessing**
2. **Data Quality Assessment and Cleaning**
3. **Seasonal Trend Decomposition (STL)**
4. **Noise Reduction Techniques**
5. **Robust Model Fitting Strategies**
6. **Best Practices and Recommendations**
7. **Model Selection and Validation**
8. **Real-world Case Study: Mobile App Adoption**

---

## 1. Introduction to Time Series Preprocessing

Real-world innovation diffusion data often contains:
- **Seasonal patterns** (holiday effects, business cycles)
- **Noise and measurement errors**
- **Missing observations** (data collection issues)
- **Outliers** (unusual events, data entry errors)
- **Irregular sampling** (uneven time intervals)

Proper preprocessing is crucial for reliable model fitting and forecasting.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Import innovate components
from innovate.diffuse.bass import BassModel
from innovate.diffuse.gompertz import GompertzModel
from innovate.diffuse.logistic import LogisticModel
from innovate.fitters.scipy_fitter import ScipyFitter
from innovate.preprocess.decomposition import stl_decomposition
from innovate.preprocess.time_series import rolling_average, sarima_fit
from innovate.utils.preprocessing import (
    ensure_datetime_index, 
    aggregate_time_series, 
    apply_rolling_average,
    apply_sarima
)
from innovate.utils.model_evaluation import get_fit_metrics, model_aic, model_bic
from innovate.plots.diffusion import plot_diffusion_curve

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
np.random.seed(42)

## 2. Data Quality Assessment and Cleaning

Let's start by generating realistic noisy time series data and learning how to assess and improve data quality.

In [ ]:
# Generate realistic noisy adoption data
def generate_realistic_adoption_data():
    """
    Generate realistic adoption data with multiple sources of noise and irregularities
    """
    # Base parameters for a Bass model
    p, q, m = 0.02, 0.35, 1000000
    
    # Create time series with monthly data over 5 years
    start_date = '2019-01-01'
    dates = pd.date_range(start=start_date, periods=60, freq='MS')  # Monthly start
    
    # Generate clean Bass adoption curve
    bass_model = BassModel(p=p, q=q, m=m)
    time_numeric = np.arange(1, len(dates) + 1)
    clean_adoptions = bass_model.predict(time_numeric)
    
    # Add realistic noise components
    
    # 1. Seasonal pattern (holiday effects)
    seasonal_effect = 0.15 * np.sin(2 * np.pi * time_numeric / 12) * clean_adoptions
    
    # 2. Random measurement noise
    noise_level = 0.08
    random_noise = np.random.normal(0, noise_level * clean_adoptions)
    
    # 3. Occasional outliers (data entry errors, unusual events)
    outlier_indices = np.random.choice(len(dates), size=5, replace=False)
    outlier_multipliers = np.ones(len(dates))
    outlier_multipliers[outlier_indices] = np.random.uniform(0.3, 2.5, size=5)
    
    # 4. Missing data (simulate 10% missing observations)
    missing_indices = np.random.choice(len(dates), size=6, replace=False)
    
    # Combine all effects
    noisy_adoptions = (clean_adoptions + seasonal_effect + random_noise) * outlier_multipliers
    noisy_adoptions = np.maximum(0, noisy_adoptions)  # Ensure non-negative
    
    # Create pandas Series
    adoption_series = pd.Series(noisy_adoptions, index=dates, name='Cumulative_Adoptions')
    
    # Introduce missing values
    adoption_series.iloc[missing_indices] = np.nan
    
    # Create clean reference for comparison
    clean_series = pd.Series(clean_adoptions, index=dates, name='Clean_Adoptions')
    
    return adoption_series, clean_series, {'p': p, 'q': q, 'm': m}

# Generate data
raw_data, clean_reference, true_params = generate_realistic_adoption_data()

print(f"Generated {len(raw_data)} data points from {raw_data.index[0]} to {raw_data.index[-1]}")
print(f"Missing values: {raw_data.isnull().sum()}")
print(f"True Bass parameters: p={true_params['p']}, q={true_params['q']}, m={true_params['m']:,}")

In [ ]:
# Data quality assessment functions
def assess_data_quality(series):
    """
    Comprehensive data quality assessment
    """
    print("=== DATA QUALITY ASSESSMENT ===")
    print(f"Total observations: {len(series)}")
    print(f"Missing values: {series.isnull().sum()} ({series.isnull().mean():.1%})")
    print(f"Zero values: {(series == 0).sum()}")
    print(f"Negative values: {(series < 0).sum()}")
    
    # Statistical summary
    print(f"\nMean: {series.mean():.2f}")
    print(f"Std Dev: {series.std():.2f}")
    print(f"Min: {series.min():.2f}")
    print(f"Max: {series.max():.2f}")
    
    # Detect outliers using IQR method
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    print(f"\nPotential outliers (IQR method): {len(outliers)}")
    if len(outliers) > 0:
        print(f"Outlier dates: {list(outliers.index.strftime('%Y-%m-%d'))}")
    
    # Check for non-decreasing property (cumulative adoption should be monotonic)
    decreasing_points = (series.diff() < -series.std() * 0.1).sum()
    print(f"\nSignificant decreases: {decreasing_points} (may indicate data issues)")
    
    return {
        'outliers': outliers,
        'missing_count': series.isnull().sum(),
        'decreasing_points': decreasing_points
    }

# Assess the raw data quality
quality_assessment = assess_data_quality(raw_data)

# Visualize raw vs clean data
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Time series plot
ax1.plot(clean_reference.index, clean_reference.values, 'b-', linewidth=2, label='True (Clean) Data', alpha=0.7)
ax1.plot(raw_data.index, raw_data.values, 'ro-', markersize=4, label='Raw (Noisy) Data', alpha=0.8)

# Highlight missing values
missing_mask = raw_data.isnull()
if missing_mask.any():
    ax1.scatter(raw_data.index[missing_mask], [0]*missing_mask.sum(), 
               color='red', s=100, marker='x', label='Missing Values')

# Highlight outliers
if len(quality_assessment['outliers']) > 0:
    ax1.scatter(quality_assessment['outliers'].index, quality_assessment['outliers'].values,
               color='orange', s=100, marker='s', label='Detected Outliers')

ax1.set_title('Raw Data Quality Assessment')
ax1.set_xlabel('Date')
ax1.set_ylabel('Cumulative Adoptions')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Residuals plot
valid_indices = ~raw_data.isnull()
residuals = raw_data[valid_indices] - clean_reference[valid_indices]
ax2.plot(residuals.index, residuals.values, 'g-', alpha=0.7, linewidth=1)
ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax2.fill_between(residuals.index, residuals.values, alpha=0.3)
ax2.set_title('Residuals (Raw - Clean Data)')
ax2.set_xlabel('Date')
ax2.set_ylabel('Residual Value')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Seasonal Trend Decomposition (STL)

STL decomposition separates the time series into trend, seasonal, and residual components, allowing us to model the underlying diffusion trend separately from seasonal effects.

In [ ]:
# Handle missing values before STL decomposition
def prepare_data_for_stl(series):
    """
    Prepare time series for STL decomposition by handling missing values
    """
    # First, interpolate missing values
    interpolated = series.interpolate(method='linear')
    
    # For any remaining NaN values at the beginning or end, use forward/backward fill
    interpolated = interpolated.fillna(method='bfill').fillna(method='ffill')
    
    return interpolated

# Prepare data and perform STL decomposition
cleaned_data = prepare_data_for_stl(raw_data)

# Apply STL decomposition using the innovate library
stl_results = stl_decomposition(cleaned_data, period=12)  # 12 months seasonality

print("STL Decomposition completed")
print(f"Components: {list(stl_results.columns)}")
print(f"Trend range: {stl_results['trend'].min():.0f} to {stl_results['trend'].max():.0f}")
print(f"Seasonal amplitude: ±{stl_results['seasonal'].abs().max():.0f}")

# Visualize STL decomposition
fig, axes = plt.subplots(4, 1, figsize=(14, 12))

# Original data
axes[0].plot(cleaned_data.index, cleaned_data.values, 'b-', linewidth=2)
axes[0].set_title('Original Time Series (Cleaned)')
axes[0].set_ylabel('Adoptions')
axes[0].grid(True, alpha=0.3)

# Trend component
axes[1].plot(stl_results.index, stl_results['trend'], 'g-', linewidth=2)
axes[1].set_title('Trend Component')
axes[1].set_ylabel('Trend')
axes[1].grid(True, alpha=0.3)

# Seasonal component
axes[2].plot(stl_results.index, stl_results['seasonal'], 'r-', linewidth=2)
axes[2].set_title('Seasonal Component')
axes[2].set_ylabel('Seasonal')
axes[2].grid(True, alpha=0.3)

# Residual component
axes[3].plot(stl_results.index, stl_results['residual'], 'orange', linewidth=1, alpha=0.7)
axes[3].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[3].set_title('Residual Component')
axes[3].set_xlabel('Date')
axes[3].set_ylabel('Residuals')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Extract deseasonalized trend for model fitting
trend_component = stl_results['trend']
deseasonalized = cleaned_data - stl_results['seasonal']

print(f"\nExtracted trend component for diffusion modeling")
print(f"Trend correlation with original: {np.corrcoef(trend_component.dropna(), cleaned_data.dropna())[0,1]:.3f}")

## 4. Noise Reduction Techniques

Apply various smoothing and filtering techniques to reduce noise while preserving the underlying diffusion pattern.

In [ ]:
# Implement various noise reduction techniques
def apply_noise_reduction_techniques(series):
    """
    Apply multiple noise reduction techniques and compare results
    """
    techniques = {}
    
    # 1. Simple moving average
    techniques['Moving Average (3)'] = apply_rolling_average(series, window=3)
    techniques['Moving Average (6)'] = apply_rolling_average(series, window=6)
    
    # 2. Exponential weighted moving average
    techniques['EWMA (α=0.3)'] = series.ewm(alpha=0.3).mean()
    techniques['EWMA (α=0.1)'] = series.ewm(alpha=0.1).mean()
    
    # 3. Savitzky-Golay filter
    from scipy.signal import savgol_filter
    window_length = min(11, len(series.dropna()) // 2 * 2 + 1)  # Must be odd
    if window_length >= 5:
        valid_data = series.dropna()
        smoothed = savgol_filter(valid_data.values, window_length, polyorder=3)
        techniques['Savitzky-Golay'] = pd.Series(smoothed, index=valid_data.index)
    
    # 4. Median filter (robust to outliers)
    techniques['Median Filter'] = series.rolling(window=5, center=True).median()
    
    # 5. Use trend component from STL as smoothed version
    techniques['STL Trend'] = trend_component
    
    return techniques

# Apply noise reduction techniques
smoothed_versions = apply_noise_reduction_techniques(cleaned_data)

# Evaluate smoothing quality
def evaluate_smoothing_quality(original, smoothed_dict, reference):
    """
    Evaluate quality of different smoothing techniques
    """
    results = {}
    
    for name, smoothed in smoothed_dict.items():
        # Align indices for comparison
        common_idx = original.index.intersection(smoothed.dropna().index)
        if len(common_idx) == 0:
            continue
            
        orig_aligned = original.loc[common_idx]
        smooth_aligned = smoothed.loc[common_idx]
        ref_aligned = reference.loc[common_idx]
        
        # Calculate metrics
        # 1. Noise reduction (how much closer to reference)
        original_rmse = np.sqrt(np.mean((orig_aligned - ref_aligned) ** 2))
        smoothed_rmse = np.sqrt(np.mean((smooth_aligned - ref_aligned) ** 2))
        noise_reduction = (original_rmse - smoothed_rmse) / original_rmse * 100
        
        # 2. Smoothness (second derivative)
        second_diff = np.diff(smooth_aligned.values, n=2)
        smoothness = -np.std(second_diff)  # Negative because lower variance = smoother
        
        # 3. Correlation with reference
        correlation = np.corrcoef(smooth_aligned, ref_aligned)[0, 1]
        
        results[name] = {
            'noise_reduction_pct': noise_reduction,
            'rmse_vs_reference': smoothed_rmse,
            'correlation_w_reference': correlation,
            'smoothness_score': smoothness
        }
    
    return results

# Evaluate smoothing techniques
smoothing_evaluation = evaluate_smoothing_quality(cleaned_data, smoothed_versions, clean_reference)

# Visualize smoothing results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot smoothed versions
ax1 = axes[0, 0]
ax1.plot(clean_reference.index, clean_reference.values, 'b-', linewidth=2, label='True (Reference)', alpha=0.7)
ax1.plot(cleaned_data.index, cleaned_data.values, 'ko', markersize=3, alpha=0.5, label='Noisy Data')

colors = ['red', 'green', 'orange', 'purple', 'brown']
for i, (name, smoothed) in enumerate(list(smoothed_versions.items())[:5]):
    if smoothed is not None and not smoothed.empty:
        ax1.plot(smoothed.index, smoothed.values, color=colors[i], linewidth=2, 
                label=name, alpha=0.8)

ax1.set_title('Comparison of Noise Reduction Techniques')
ax1.set_xlabel('Date')
ax1.set_ylabel('Cumulative Adoptions')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Evaluation metrics heatmap
ax2 = axes[0, 1]
eval_df = pd.DataFrame(smoothing_evaluation).T
eval_df_normalized = (eval_df - eval_df.min()) / (eval_df.max() - eval_df.min())

sns.heatmap(eval_df_normalized, annot=True, cmap='RdYlGn', ax=ax2, fmt='.2f')
ax2.set_title('Smoothing Technique Evaluation\n(Normalized Metrics)')

# Residuals comparison
ax3 = axes[1, 0]
for i, (name, smoothed) in enumerate(list(smoothed_versions.items())[:3]):
    if smoothed is not None and not smoothed.empty:
        common_idx = smoothed.index.intersection(clean_reference.index)
        residuals = smoothed.loc[common_idx] - clean_reference.loc[common_idx]
        ax3.plot(residuals.index, residuals.values, color=colors[i], 
                linewidth=1.5, alpha=0.7, label=name)

ax3.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax3.set_title('Residuals vs Reference (Top 3 Methods)')
ax3.set_xlabel('Date')
ax3.set_ylabel('Residual')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Performance summary
ax4 = axes[1, 1]
methods = list(smoothing_evaluation.keys())
noise_reductions = [smoothing_evaluation[m]['noise_reduction_pct'] for m in methods]

bars = ax4.bar(range(len(methods)), noise_reductions, color=colors[:len(methods)])
ax4.set_title('Noise Reduction Performance')
ax4.set_xlabel('Method')
ax4.set_ylabel('Noise Reduction (%)')
ax4.set_xticks(range(len(methods)))
ax4.set_xticklabels(methods, rotation=45, ha='right')

# Add value labels on bars
for i, (bar, value) in enumerate(zip(bars, noise_reductions)):
    ax4.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{value:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\n=== SMOOTHING EVALUATION RESULTS ===")
eval_summary_df = pd.DataFrame(smoothing_evaluation).T
print(eval_summary_df.round(3))

# Select best smoothed version for model fitting
best_method = max(smoothing_evaluation.keys(), 
                 key=lambda x: smoothing_evaluation[x]['correlation_w_reference'])
best_smoothed = smoothed_versions[best_method]
print(f"\nBest smoothing method selected: {best_method}")

## 5. Robust Model Fitting Strategies

Fit diffusion models to the preprocessed data using robust techniques and compare different approaches.

In [ ]:
# Robust model fitting with multiple approaches
def fit_multiple_models(time_series):
    """
    Fit multiple diffusion models to the time series and compare results
    """
    # Prepare numeric time points
    valid_data = time_series.dropna()
    time_points = np.arange(1, len(valid_data) + 1)
    adoption_values = valid_data.values
    
    models = {}
    fitter = ScipyFitter()
    
    # 1. Bass Model
    try:
        bass = BassModel()
        fitter.fit(bass, time_points, adoption_values)
        models['Bass'] = bass
    except Exception as e:
        print(f"Bass model fitting failed: {e}")
    
    # 2. Logistic Model
    try:
        logistic = LogisticModel()
        fitter.fit(logistic, time_points, adoption_values)
        models['Logistic'] = logistic
    except Exception as e:
        print(f"Logistic model fitting failed: {e}")
    
    # 3. Gompertz Model
    try:
        gompertz = GompertzModel()
        fitter.fit(gompertz, time_points, adoption_values)
        models['Gompertz'] = gompertz
    except Exception as e:
        print(f"Gompertz model fitting failed: {e}")
    
    return models, time_points, adoption_values

# Fit models to different preprocessed versions
preprocessing_approaches = {
    'Raw (Cleaned)': cleaned_data,
    'STL Trend': trend_component,
    'Best Smoothed': best_smoothed
}

all_fitting_results = {}

for approach_name, data_version in preprocessing_approaches.items():
    print(f"\nFitting models to: {approach_name}")
    models, time_pts, adoption_vals = fit_multiple_models(data_version)
    all_fitting_results[approach_name] = {
        'models': models,
        'time_points': time_pts,
        'adoption_values': adoption_vals,
        'data_series': data_version
    }
    
    print(f"Successfully fitted {len(models)} models")

# Model evaluation and comparison
def evaluate_model_fits(fitting_results, reference_params):
    """
    Evaluate and compare model fits across different preprocessing approaches
    """
    evaluation_results = {}
    
    for approach_name, results in fitting_results.items():
        models = results['models']
        time_points = results['time_points']
        adoption_values = results['adoption_values']
        
        approach_results = {}
        
        for model_name, model in models.items():
            # Calculate fit metrics
            try:
                fit_metrics = get_fit_metrics(model, time_points, adoption_values)
                
                # Parameter comparison with true values (for Bass model)
                param_accuracy = {}
                if model_name == 'Bass' and hasattr(model, 'params_') and model.params_:
                    for param in ['p', 'q', 'm']:
                        if param in model.params_:
                            true_val = reference_params[param]
                            fitted_val = model.params_[param]
                            error_pct = abs(fitted_val - true_val) / true_val * 100
                            param_accuracy[f'{param}_error_pct'] = error_pct
                
                # AIC and BIC
                try:
                    aic_score = model_aic(model, time_points, adoption_values)
                    bic_score = model_bic(model, time_points, adoption_values)
                    fit_metrics['AIC'] = aic_score
                    fit_metrics['BIC'] = bic_score
                except:
                    pass
                
                approach_results[model_name] = {
                    'fit_metrics': fit_metrics,
                    'param_accuracy': param_accuracy,
                    'parameters': model.params_
                }
                
            except Exception as e:
                print(f"Error evaluating {model_name} in {approach_name}: {e}")
        
        evaluation_results[approach_name] = approach_results
    
    return evaluation_results

# Evaluate all model fits
model_evaluations = evaluate_model_fits(all_fitting_results, true_params)

print("\n=== MODEL EVALUATION COMPLETED ===")

In [ ]:
# Visualize model fitting results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Model fits for different preprocessing approaches
ax1 = axes[0, 0]
colors_models = {'Bass': 'red', 'Logistic': 'blue', 'Gompertz': 'green'}
linestyles = {'Raw (Cleaned)': '-', 'STL Trend': '--', 'Best Smoothed': '-.'}

# Plot reference data
ax1.plot(clean_reference.index, clean_reference.values, 'k-', linewidth=3, 
         alpha=0.5, label='True Data')

# Plot fitted models
for approach_name, results in all_fitting_results.items():
    models = results['models']
    time_points = results['time_points']
    data_series = results['data_series']
    
    # Convert time points back to dates for plotting
    valid_dates = data_series.dropna().index
    
    for model_name, model in models.items():
        if hasattr(model, 'params_') and model.params_:
            predictions = model.predict(time_points)
            ax1.plot(valid_dates, predictions, 
                    color=colors_models[model_name], 
                    linestyle=linestyles[approach_name],
                    linewidth=2, alpha=0.8,
                    label=f'{model_name} ({approach_name})')

ax1.set_title('Model Fits Across Preprocessing Approaches')
ax1.set_xlabel('Date')
ax1.set_ylabel('Cumulative Adoptions')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)

# Plot 2: R² comparison
ax2 = axes[0, 1]
r_squared_data = []
approach_labels = []
model_labels = []

for approach_name, approach_results in model_evaluations.items():
    for model_name, model_result in approach_results.items():
        if 'R_squared' in model_result['fit_metrics']:
            r_squared_data.append(model_result['fit_metrics']['R_squared'])
            approach_labels.append(approach_name)
            model_labels.append(model_name)

# Create grouped bar chart
x_pos = np.arange(len(r_squared_data))
colors_bars = [colors_models.get(model, 'gray') for model in model_labels]
bars = ax2.bar(x_pos, r_squared_data, color=colors_bars, alpha=0.7)

ax2.set_title('Model Fit Quality (R²)')
ax2.set_ylabel('R² Score')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'{m}\n{a}' for m, a in zip(model_labels, approach_labels)], 
                   rotation=45, ha='right')

# Add value labels on bars
for bar, value in zip(bars, r_squared_data):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{value:.3f}', ha='center', va='bottom')

ax2.grid(True, alpha=0.3)

# Plot 3: Parameter accuracy for Bass model
ax3 = axes[1, 0]
bass_param_errors = []
bass_approaches = []
param_names = ['p', 'q', 'm']

for approach_name, approach_results in model_evaluations.items():
    if 'Bass' in approach_results:
        param_accuracy = approach_results['Bass']['param_accuracy']
        for param in param_names:
            error_key = f'{param}_error_pct'
            if error_key in param_accuracy:
                bass_param_errors.append(param_accuracy[error_key])
                bass_approaches.append(f'{param}\n{approach_name}')

if bass_param_errors:
    x_pos = np.arange(len(bass_param_errors))
    param_colors = ['red', 'green', 'blue'] * (len(bass_param_errors) // 3 + 1)
    bars = ax3.bar(x_pos, bass_param_errors, color=param_colors[:len(bass_param_errors)], alpha=0.7)
    
    ax3.set_title('Bass Model Parameter Accuracy')
    ax3.set_ylabel('Error (%)')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(bass_approaches, rotation=45, ha='right')
    
    for bar, value in zip(bars, bass_param_errors):
        ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{value:.1f}%', ha='center', va='bottom')

ax3.grid(True, alpha=0.3)

# Plot 4: AIC comparison
ax4 = axes[1, 1]
aic_data = []
aic_labels = []

for approach_name, approach_results in model_evaluations.items():
    for model_name, model_result in approach_results.items():
        if 'AIC' in model_result['fit_metrics']:
            aic_data.append(model_result['fit_metrics']['AIC'])
            aic_labels.append(f'{model_name}\n{approach_name}')

if aic_data:
    x_pos = np.arange(len(aic_data))
    colors_bars = [colors_models.get(label.split('\n')[0], 'gray') for label in aic_labels]
    bars = ax4.bar(x_pos, aic_data, color=colors_bars, alpha=0.7)
    
    ax4.set_title('Model Selection (AIC - Lower is Better)')
    ax4.set_ylabel('AIC Score')
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels(aic_labels, rotation=45, ha='right')
    
    for bar, value in zip(bars, aic_data):
        ax4.text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(aic_data) * 0.01,
                 f'{value:.0f}', ha='center', va='bottom')

ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print detailed results summary
print("\n=== DETAILED FITTING RESULTS SUMMARY ===")
for approach_name, approach_results in model_evaluations.items():
    print(f"\n{approach_name.upper()}:")
    for model_name, model_result in approach_results.items():
        print(f"  {model_name} Model:")
        if 'R_squared' in model_result['fit_metrics']:
            print(f"    R² = {model_result['fit_metrics']['R_squared']:.4f}")
        if 'RMSE' in model_result['fit_metrics']:
            print(f"    RMSE = {model_result['fit_metrics']['RMSE']:.2f}")
        if 'AIC' in model_result['fit_metrics']:
            print(f"    AIC = {model_result['fit_metrics']['AIC']:.2f}")
        
        # Parameter accuracy for Bass model
        if model_name == 'Bass' and model_result['param_accuracy']:
            print(f"    Parameter Errors:")
            for param_error, value in model_result['param_accuracy'].items():
                print(f"      {param_error}: {value:.1f}%")

## 6. Best Practices and Recommendations

Based on our comprehensive analysis, here are the key takeaways for time series preprocessing and robust model fitting:

### Preprocessing Best Practices:

1. **Always Assess Data Quality First**: Check for missing values, outliers, and non-monotonic behavior
2. **Use STL Decomposition**: Separate trend from seasonal effects for cleaner model fitting
3. **Apply Appropriate Smoothing**: STL trend or EWMA often work best for innovation diffusion data
4. **Validate Preprocessing**: Ensure smoothed data maintains key characteristics of the diffusion process

### Model Fitting Strategies:

1. **Try Multiple Models**: Compare Bass, Logistic, and Gompertz models
2. **Use Information Criteria**: AIC/BIC help select the best model for your data
3. **Cross-validate Results**: Test model performance on holdout data
4. **Consider Parameter Interpretability**: Ensure fitted parameters make business sense

### When to Use Each Approach:

- **Raw Data**: When data quality is high and noise is minimal
- **STL Trend**: When strong seasonal patterns need to be removed
- **Smoothed Data**: When data contains significant measurement noise
- **Multiple Approaches**: Always compare to ensure robustness

This tutorial provides a comprehensive framework for handling real-world innovation diffusion data, ensuring reliable and interpretable results from your modeling efforts.